# 🛠️ Setup e Carga


In [0]:
#Importando
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [0]:
#Carregar as tabelas que foram subidas
#df = spark.table("base_tratada_1").toPandas()
df = pd.read_csv("data/Inputs/base_tratada.csv")

# 🚚 Módulo 1: Logística e SLA

In [0]:
#Garantir que as colunas de data sejam lidas como data
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_carrier_date'] = pd.to_datetime(df['order_delivered_carrier_date'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])

In [0]:
# 1. Calcula a diferença em dias entre a compra e a postagem
df['dias_postagem'] = (df['order_delivered_carrier_date'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400

# 2. Define a função que classifica o vendedor
def categorizar_postagem(dias):
    if dias <= 1: 
        return '0-1 dia (Excelente)'
    elif dias <= 3: 
        return '2-3 dias (Bom)'
    else: 
        return 'Mais de 3 dias (Gargalo)'

# 3. Aplica a função criando a nova coluna 'eficiencia_vendedor'
df['eficiencia_vendedor'] = df['dias_postagem'].apply(categorizar_postagem)

# 4. Define a ordem fixa para o gráfico não ficar bagunçado
order_list = ['0-1 dia (Excelente)', '2-3 dias (Bom)', 'Mais de 3 dias (Gargalo)']

In [0]:
plt.figure(figsize=(10, 6))

# Cores: Verde (Sucesso), Azul (Neutro), Vermelho (Alerta/Gargalo)
cores = ['#2ecc71', '#3498db', '#e74c3c']

# Cria o gráfico de barras
ax = sns.countplot(data=df, x='eficiencia_vendedor', order=order_list, palette=cores)

# Adiciona os números exatos acima de cada barra
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points',
                fontsize=11, fontweight='bold')

# Títulos e rótulos
plt.title('Análise de Gargalo: Eficiência de Postagem dos Vendedores', fontsize=14, pad=20)
plt.xlabel('Tempo para entrega à transportadora', fontsize=12)
plt.ylabel('Quantidade de Pedidos', fontsize=12)

# Remove as bordas do gráfico para ficar mais limpo
sns.despine()

plt.show()

**Insight 01: Logísitica e SLA**

- **Diagnóstico**: Identificamos que mais de 40.000 pedidos levam mais de 3 dias para serem postados (Gargalo).
- **Impacto na Receita**: Vendedores lentos aumentam o churn (cancelamento) e diminuem a nota de satisfação, impedindo o crescimento orgânico.
- **Recomendação**: Implementar uma regra de negócio que priorize vendedores com postagem <24h no algoritmo de busca da plataforma.

In [0]:
# 1. Criar uma tabela cruzada (Cross-tab) entre eficiência e se houve atraso
cruzamento = pd.crosstab(df['eficiencia_vendedor'], df['is_delayed'], normalize='index') * 100

# 2. Plotar o gráfico de barras empilhadas (Stacked Bar)
cruzamento.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#2ecc71', '#e74c3c'])

# Configurações do gráfico
plt.title('Impacto da Eficiência do Vendedor no Atraso Final (SLA)', fontsize=14)
plt.xlabel('Categoria de Eficiência do Vendedor')
plt.ylabel('Percentual de Pedidos (%)')
plt.legend(title='Atrasou?', labels=['No Prazo', 'Atrasado'], loc='upper right')
plt.xticks(rotation=0)
sns.despine()

plt.show()

**Insight 02: Sensibilidade ao Atraso**
**Observação Crítica**: Embora a maioria dos pedidos chegue no prazo, o grupo de vendedores com Gargalo de Postagem apresenta uma taxa de atraso final significativamente superior aos demais.

**O Risco**: O tempo de postagem consome a "margem de erro" da transportadora. Vendedores lentos tornam a operação logística vulnerável.

**Conclusão:** Não é um cenário de crise total, mas é um ponto de ineficiência. Reduzir o gargalo de postagem é a forma mais barata de aumentar a segurança do SLA sem gastar mais com frete expresso.

In [0]:
# Gráfico de barras comparando a nota média de satisfação
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df, x='eficiencia_vendedor', y='review_score_mean', order=order_list, palette='RdYlGn')

# Adicionar a linha da média geral para comparação
media_geral = df['review_score_mean'].mean()
plt.axhline(media_geral, color='black', linestyle='--', label=f'Média Geral: {media_geral:.2f}')

plt.title('Impacto do Atraso de Postagem na Nota do Cliente')
plt.ylabel('Média de Satisfação (1 a 5)', fontsize=12)
plt.xlabel('Eficiência do Vendedor', fontsize=12)
plt.ylim(0, 5) # Notas de 0 a 5
plt.legend()
plt.show()

**Insight 03: Experiência do Cliente vs. Agilidade na Origem**
**O Fato**: Existe uma queda direta na satisfação do cliente conforme o vendedor demora a postar. Vendedores na categoria Excelente mantêm notas acima da média da plataforma (> 4.08).

**O Alerta**: O grupo Gargalo é o único que fica abaixo da média geral. Isso prova que a insatisfação começa antes mesmo do transporte, no "tempo de espera" do processamento do pedido.

**Impacto Financeiro**: Notas baixas diminuem a taxa de recompra. Melhorar a postagem não é apenas logística, é estratégia de retenção de receita.

**Ação**: Vendedores com média abaixo de 4.0 e postagem acima de 3 dias devem entrar em um plano de melhoria compulsória.

In [0]:
# Cálculo de faturamento por grupo de eficiência
receita_por_grupo = df.groupby('eficiencia_vendedor')['item_total'].sum().reindex(order_list)
print("Faturamento por Categoria de Eficiência:")
print(receita_por_grupo.apply(lambda x: f"R$ {x:,.2f}"))

**Insight 04**: Análise de Receita Exposta ao Risco
**O Fato**: O faturamento não está concentrado nos vendedores mais eficientes. Pelo contrário, o grupo Gargalo (postagem > 3 dias) movimenta o maior volume financeiro: R$ 6.393.510,48.

**A Gravidade**: Isso mostra que a Olist depende de vendedores operacionalmente ineficientes para sustentar seu faturamento atual.

**O Risco de Crescimento**: Como esses vendedores têm as piores notas, a chance de esses clientes (que geraram os R$ 6,3M) não voltarem a comprar é altíssima.

**Ação Estratégica**: Não podemos simplesmente banir esses vendedores (perderíamos muita receita), mas precisamos de um plano de aceleração urgente para converter esse faturamento do grupo "Gargalo" para o grupo "Excelente".

**"A Postagem é o Primeiro Momento da Verdade." Se o vendedor falha ali, a transportadora pode ser a mais rápida do mundo, mas o cliente já estará inclinado a dar uma nota menor porque a ansiedade da compra foi frustrada logo no início.**

# 💰 Módulo 2: Crescimento e Receita
Nesta seção, analisamos a saúde financeira da plataforma, identificando as categorias que sustentam o faturamento e as oportunidades de expansão através da Regra de Pareto (80/20).

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Agrupar faturamento por categoria e ordenar do maior para o menor
pareto_df = df.groupby('product_category')['item_total'].sum().reset_index()
pareto_df = pareto_df.sort_values(by='item_total', ascending=False)

# 2. Calcular a porcentagem individual e a acumulada (importante para o Pareto)
pareto_df['percentual'] = (pareto_df['item_total'] / pareto_df['item_total'].sum()) * 100
pareto_df['acumulado'] = pareto_df['percentual'].cumsum()

# 3. Configurar o gráfico com dois eixos (Eixo Y1: Reais | Eixo Y2: Porcentagem)
fig, ax1 = plt.subplots(figsize=(15, 8))

# Gráfico de Barras (Faturamento por Categoria)
sns.barplot(data=pareto_df.head(20), x='product_category', y='item_total', ax=ax1, palette='mako')
ax1.set_title('Curva de Pareto: Top 20 Categorias por Faturamento', fontsize=16, pad=20)
ax1.set_ylabel('Faturamento Total (R$)', fontsize=12)
ax1.set_xlabel('Categorias de Produtos', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Linha de Pareto (Percentual Acumulado)
ax2 = ax1.twinx()
ax2.plot(pareto_df.head(20)['product_category'], pareto_df.head(20)['acumulado'], color='red', marker='D', ms=7, label='Acumulado %')
ax2.axhline(80, color='orange', linestyle='--', linewidth=2, label='Linha de 80%') # A linha mágica do 80/20
ax2.set_ylabel('Percentual Acumulado (%)', fontsize=12)
ax2.set_ylim(0, 110)

# Ajustes finais de layout
sns.despine(right=False)
plt.tight_layout()
plt.show()

%md
### 🎯 Insight 05: Foco Estratégico (Regra de Pareto 80/20)

* **Diagnóstico de Concentração:** O gráfico revela que a receita da Olist é extremamente dependente de um grupo seleto de categorias. A "Linha de 80%" (laranja) é atingida rapidamente, destacando que as primeiras **14 categorias** (de *Beleza/Saúde* até *Telefonia*) sustentam quase todo o faturamento da plataforma.
* **A "Cauda Longa":** As demais categorias à direita representam uma variedade importante para o catálogo, mas possuem baixo impacto financeiro imediato.
* **Estratégia de Crescimento:**
    1.  **Prioridade Máxima:** Qualquer esforço para reduzir o **Gargalo de Postagem** deve ser focado primeiro nessas 14 categorias vitais. Um ganho de 1% aqui vale mais do que 50% nas categorias menores.
    2.  **Eficiência Logística:** Se garantirmos que as categorias *health_beauty* e *watches_gifts* (líderes de faturamento) operem no nível "Excelente" de postagem, protegemos quase **30% da receita total** contra avaliações negativas e churn (perda de clientes).

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- GARANTIA DE DADOS (Caso a variável tenha se perdido) ---
# 1. Calcula a média por categoria
ticket_medio = df.groupby('product_category')['item_total'].mean().reset_index()

# 2. Pega as categorias do Top 20 do faturamento (usando o seu pareto_df)
top_20_categorias = pareto_df.head(20)['product_category']
ticket_medio_top = ticket_medio[ticket_medio['product_category'].isin(top_20_categorias)]
ticket_medio_top = ticket_medio_top.sort_values(by='item_total', ascending=False)
# ------------------------------------------------------------

# 1. Ajuste do tamanho da figura para dar respiro lateral
plt.figure(figsize=(16, 8)) 

# 2. Criação do gráfico
ax = sns.barplot(data=ticket_medio_top, x='product_category', y='item_total', palette='viridis')

# 3. AJUSTE DOS RÓTULOS
for p in ax.patches:
    altura = p.get_height()
    ax.annotate(f'R$ {altura:.2f}', 
                (p.get_x() + p.get_width() / 2., altura), 
                ha = 'left', va = 'bottom',
                xytext = (3, 5),
                textcoords = 'offset points',
                fontsize=9, 
                fontweight='bold',
                rotation=45)

# 4. Títulos e Eixos
plt.title('Ticket Médio por Categoria (Top 20 Faturamento)', fontsize=16, pad=35)
plt.ylabel('Valor Médio por Pedido (R$)', fontsize=12)
plt.xlabel('Categorias de Produtos', fontsize=12)
plt.xticks(rotation=45, ha='right')

# 5. Margem de segurança no topo
plt.ylim(0, ticket_medio_top['item_total'].max() * 1.25)

sns.despine()
plt.tight_layout()
plt.show()

%md
### 💳 Insight 06: Disparidade de Valor Agregado vs. Logística
* **O Gigante do Ticket:** A categoria de **Computadores** (*computers*) lidera com um ticket médio superior a **R$ 1.100,00**. Isso indica que cada erro logístico nesta categoria custa caro para a empresa em termos de estorno e insatisfação de clientes de alto valor.
* **Volume vs. Valor:** Categorias como *Beleza* (*health_beauty*) e *Relógios* (*watches_gifts*) sustentam o faturamento pelo volume, com tickets médios entre R$ 150 e R$ 220. 
* **Estratégia Recomendada:** 1. **Produtos High-Ticket:** Implementar seguro obrigatório e prioridade de postagem para categorias acima de R$ 500,00.
    2. **Produtos de Massa:** Focar em automação de etiquetas para vendedores de Beleza, visando reduzir o tempo de postagem sem aumentar o custo fixo.

%md
# 🏁 Resumo Executivo e Plano de Ação

Após a análise exploratória dos dados, consolidamos os seguintes pontos críticos para a diretoria:

### 1. O Diagnóstico da Operação
Identificamos que a **agilidade do vendedor** é o principal "ping" da nossa rede. Vendedores com mais de 3 dias para postar (Grupo Gargalo) possuem uma taxa de atraso final significativamente maior e, consequentemente, as piores notas de satisfação (abaixo da média geral de 4.08).

### 2. O Risco Financeiro Exposto
O dado mais alarmante é que o maior faturamento da plataforma (**R$ 6.3 milhões**) está concentrado justamente no grupo de vendedores ineficientes (Gargalo). Estamos colocando nossa maior fatia de receita em risco devido à má experiência de entrega.

### 3. Priorização por Pareto (80/20)
Não precisamos consertar tudo de uma vez. Apenas **14 categorias** geram 80% da receita. Focando a melhoria logística em *Beleza*, *Relógios* e *Informática*, protegemos a saúde financeira da empresa com o menor esforço operacional possível.

### 🚀 Próximos Passos Propostos:
* **Campanha de Aceleração:** Bonificar vendedores das Top 14 categorias que migrarem do status "Gargalo" para "Excelente".
* **Monitoramento N2:** Criar um alerta automático para o time de suporte sempre que um pedido de alto ticket (Computadores) ultrapassar 48h sem postagem.
* **Revisão de SLA:** Ajustar a promessa de entrega no site baseando-se na eficiência real do vendedor capturada nesta análise.

Observação: O dado nos mostrou que a Olist não tem um problema de transporte, ela tem um problema de processamento na origem, e isso está custando caro para as categorias que mais trazem dinheiro para a casa.